# Which nacelle parts can a robot team replace offshore? The replacement matrix

**Part of the *Robotics for Offshore Wind* Virtual Research Building (AICOR, University of Bremen), prepared for the Siemens Energy Innovation Lab, Cuxhaven.**

A service visit without people in the turbine needs a way to get every spare part in and every old part out. This
notebook takes 20 selected nacelle components of an SG 14-236 DD and checks four routes for each:

| Route | How the part travels | Limit in the twin |
|---|---|---|
| **R** | carried by the legged robot | payload 15 kg (ANYmal D, public spec) |
| **D** | cargo drone to the roof pad, then down the kit shaft | payload 100 kg (public spec), shaft 0.9 × 0.9 m |
| **H** | service-vessel crane to the platform, nacelle hoist through the hatch, robotic bridge crane | hatch 1.4 × 0.9 m, hoist capacity |
| **X** | external heavy lift (jack-up or large crane) | everything else |

A route works if the mass fits, the part passes the opening (its two smaller dimensions fit; the longest can hang
vertically), and the work point can be reached on that route's level.

**The point for design.** In this matrix the nacelle's openings and hoist decide whether robots can service the
turbine at all. Those openings are fixed in Cuxhaven, at design time. Change them below and see what changes.

**Read the results with these limits in mind.**
- The twin is **reconstructed from public and second-hand sources**. It is not Siemens Energy design data.
- **Several masses and dimensions are illustrative.** The table marks which ones.
- **The 20 components are a selection**, not the full list of replaceable units.

In [ ]:
import os, sys, json, subprocess
from IPython.display import HTML, display
TWIN = os.path.join(os.environ.get("REPO_DIR", ".."), "CuxhavenSiteTwin")
if not os.path.isdir(TWIN):
    TWIN = os.path.abspath(os.path.join("..", "CuxhavenSiteTwin"))
sys.path.insert(0, os.path.join(TWIN, "offshore"))
import replacement_matrix as RM          # the twin's model (offshore/replacement_matrix.py)
commit = subprocess.run(["git", "-C", TWIN, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("twin commit", commit, "|", len(RM.LRU), "components | hoist hatch", RM.HATCH, "m | kit shaft", RM.KIT, "m")

## 1. The same rules, with the limits as parameters

The twin's `route_eval` has the openings and payloads built in. The function below uses the same rules but takes
the limits as arguments. The cell checks that, with the twin's values, it gives exactly the twin's answer for every
component and capacity.

In [ ]:
DEFAULT = dict(hatch=RM.HATCH, kit=RM.KIT, drone_t=0.1, robot_t=0.015, cap_t=2.0)

def evaluate(name, mass, dims, zone, hatch, kit, drone_t, robot_t, cap_t):
    r = {}
    r["R"] = mass <= robot_t and zone in ("lower", "upper", "gen", "yaw")
    r["D"] = mass <= drone_t and RM.fits(dims, kit) and zone in ("lower", "upper", "roof")
    hoist_ok = mass <= cap_t and RM.fits(dims, hatch)
    r["H"] = hoist_ok and zone == "lower"
    if hoist_ok and zone == "upper":
        r["H"] = RM.fits(dims, kit)
    return "R" if r["R"] else "D" if r["D"] else "H" if r["H"] else "X"

same = all(evaluate(n, m, d, z, **dict(DEFAULT, cap_t=c)) == RM.route_eval(n, m, d, z, c)["best"]
           for n, (m, d, z, ds, ms) in RM.LRU.items() for c in RM.CAPS)
print("identical to the twin's route_eval for all components and capacities:", same)

In [ ]:
COL = {"R": "#1f6f6b", "D": "#2b5c9a", "H": "#b87400", "X": "#a83232"}
LAB = {"R": "robot carries", "D": "drone + kit shaft", "H": "hoist + robotic crane", "X": "external heavy lift"}

def matrix(**p):
    lim = dict(DEFAULT, **p)
    rows = [(n, m, d, z, ds, ms, evaluate(n, m, d, z, **lim)) for n, (m, d, z, ds, ms) in RM.LRU.items()]
    counts = {b: sum(1 for r in rows if r[6] == b) for b in "RDHX"}
    return rows, counts, lim

def show(rows, counts, lim, base=None):
    head = (f"<p><b>Hatch {lim['hatch'][0]:.2f} × {lim['hatch'][1]:.2f} m, kit shaft {lim['kit'][0]:.2f} × {lim['kit'][1]:.2f} m, "
            f"drone ≤ {lim['drone_t']*1000:.0f} kg, robot ≤ {lim['robot_t']*1000:.0f} kg, hoist ≤ {lim['cap_t']:g} t.</b> "
            + " · ".join(f"<span style='color:{COL[b]}'><b>{b}</b> {LAB[b]}: {counts[b]}</span>" for b in "RDHX")
            + f" · <b>without external lift: {counts['R'] + counts['D'] + counts['H']} of {len(rows)}</b></p>")
    t = ["<table style='font-size:12px;border-collapse:collapse'><tr><th align=left>component</th><th>mass</th>"
         "<th>size [m]</th><th>route</th><th align=left>inputs</th></tr>"]
    for (n, m, d, z, ds, ms, b), old in zip(rows, base or [None] * len(rows)):
        changed = old is not None and old[6] != b
        mass = f"{m*1000:.0f} kg" if m < 1 else f"{m:.1f} t"
        flag = "; ".join(x for x in (f"mass {ms}" if ms == "ILLUSTRATIVE" else "", f"size {ds}" if ds == "ILLUSTRATIVE" else "") if x)
        t.append(f"<tr style='border-top:1px solid #ccc'><td>{n}</td><td align=right>{mass}</td>"
                 f"<td>{' × '.join(f'{x:.2f}' for x in d)}</td>"
                 f"<td align=center style='background:{COL[b]};color:white;font-weight:bold'>{b}{' ●' if changed else ''}</td>"
                 f"<td style='color:#777'>{flag}</td></tr>")
    t.append("</table><p style='font-size:11px;color:#777'>● = route changed against the twin's design.</p>")
    display(HTML(head + "".join(t)))

BASE = matrix()
show(*BASE)

## 2. The twin's three hoist capacities

The twin evaluates the hoist at 0.5, 1 and 2 t.

In [ ]:
for c in RM.CAPS:
    rows, counts, _ = matrix(cap_t=c)
    print(f"hoist {c:g} t:", counts, "-> without external lift:", counts["R"] + counts["D"] + counts["H"], "of", len(rows))

**What to look for.** A bigger hoist alone barely helps, because most heavy parts do not fit through the hatch or
sit where no in-nacelle route reaches them (yaw deck, generator, hub). The bottleneck is the openings and access, not
lifting capacity.

## 3. Design the nacelle for robot service

Change the openings and limits and press **Run**. Changed routes are marked. Without widgets, edit the last cell.

In [ ]:
try:
    import ipywidgets as W
    s = lambda v, lo, hi, st, d: W.FloatSlider(value=v, min=lo, max=hi, step=st, description=d, style={"description_width": "120px"})
    w = dict(hw=s(1.4, 0.8, 3.5, 0.05, "Hatch width m"), hd=s(0.9, 0.6, 3.0, 0.05, "Hatch depth m"),
             k=s(0.9, 0.6, 1.6, 0.05, "Kit shaft m"), drone=s(100, 25, 300, 25, "Drone kg"),
             robot=s(15, 5, 60, 5, "Robot kg"), cap=s(2.0, 0.5, 30.0, 0.5, "Hoist t"))
    out = W.Output(); btn = W.Button(description="Run", button_style="primary")
    def go(_):
        v = {k: x.value for k, x in w.items()}
        res = matrix(hatch=(v["hw"], v["hd"]), kit=(v["k"], v["k"]), drone_t=v["drone"] / 1000, robot_t=v["robot"] / 1000, cap_t=v["cap"])
        with out:
            out.clear_output(); show(*res, base=BASE[0])
    btn.on_click(go)
    display(W.VBox(list(w.values()) + [btn, out]))
except ImportError:
    print("ipywidgets is not installed; use the next cell")

In [ ]:
# Without widgets: a larger hatch (3.0 x 2.6 m) and a 25 t hoist
show(*matrix(hatch=(3.0, 2.6), cap_t=25.0), base=BASE[0])

## 4. What this lab cannot tell you

- **Whether the work at the part can be done by a robot.** Unbolting, cable mating and alignment need physics. The twin has one such
  episode, the coolant filter change, which succeeds on a redesigned pump skid and fails on the skid as built.
- **Weather.** Drone and hoist limits on wind, and gangway limits on wave height, decide whether a route can be used on a given day.
- **Real component data.** Masses, sizes and mounting positions are the twin's estimates; the table marks the illustrative ones.

Model: `CuxhavenSiteTwin/offshore/replacement_matrix.py`. Masses: `CuxhavenSiteTwin/product/catalog.py`. Openings and access:
`CuxhavenSiteTwin/offshore/service_access.py`.